# Cross-Domain Validation

In [1]:
# config

DOMAIN = "games" # or movies
DATA_FILE = "games_big_nli.parquet"

NLI_COLS = [
    "nli_raw_gameplay",
    "nli_raw_story",
    "nli_raw_graphics",
]

SEED          = 42
K             = 10
EMB_DIM       = 32
ASP_DIM       = 16
MLP_LAYERS    = [64, 32, 16]
DROPOUT       = 0.1
BATCH_SIZE    = 256
LR_SCRATCH    = 1e-3
LR_WARMSTART  = 3e-4
WEIGHT_DECAY  = 1e-5
LAMBDA_SENT   = 0.005
MAX_EPOCHS    = 50
PATIENCE      = 4
LGCN_LAYERS   = 1


In [2]:
# imports and global setup
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from pathlib import Path
from tqdm import tqdm
import random, json, scipy.sparse as sp
from IPython.display import display, HTML

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"Domain  : {DOMAIN}")
print(f"Device  : {DEVICE}")
print(f"NLI cols: {NLI_COLS}")

Domain  : games
Device  : cuda
NLI cols: ['nli_raw_gameplay', 'nli_raw_story', 'nli_raw_graphics']


In [ ]:
# Data Validation

OK   = "\033[92m  OK   \033[0m"
FAIL = "\033[91m  FAIL \033[0m"
WARN = "\033[93m  WARN \033[0m"


candidates = list(Path("/kaggle/input").rglob(DATA_FILE))
if not candidates:
    print(f"{FAIL} File not found: {DATA_FILE}")
    raise FileNotFoundError(DATA_FILE)
data_path = candidates[0]
print(f"{OK} File found: {data_path}")

df_val = pd.read_parquet(data_path)
print(f"{OK} Loaded — {len(df_val):,} rows, {df_val['user_id'].nunique():,} users, {df_val['item_id'].nunique():,} items")
print(f"     Columns: {df_val.columns.tolist()}")

BASE_COLS = ["user_id", "item_id", "timestamp", "sentiment_score"]
missing_base = [c for c in BASE_COLS if c not in df_val.columns]
if missing_base:
    print(f"{FAIL} Missing base columns: {missing_base}")
else:
    print(f"{OK} All base columns present: {BASE_COLS}")

missing_nli = [c for c in NLI_COLS if c not in df_val.columns]
if missing_nli:
    print(f"{FAIL} Missing NLI columns: {missing_nli}")
    print(f"       Available columns containing 'nli': {[c for c in df_val.columns if 'nli' in c.lower()]}")
else:
    print(f"{OK} All NLI columns present: {NLI_COLS}")

print(f"\n  --- NaN rates ---")
for col in BASE_COLS + NLI_COLS:
    if col not in df_val.columns:
        continue
    rate = df_val[col].isna().mean() * 100
    status = OK if rate < 50 else WARN
    print(f"  {status} {col:<35} {rate:5.1f}% missing")

print(f"\n  --- NLI value ranges ---")
for col in NLI_COLS:
    if col not in df_val.columns:
        continue
    s = df_val[col].dropna()
    if len(s) == 0:
        print(f"  {FAIL} {col}: all NaN")
        continue
    out_of_range = ((s < -1.5) | (s > 1.5)).sum()
    status = OK if out_of_range == 0 else WARN
    print(f"  {status} {col:<35} min={s.min():.3f}  max={s.max():.3f}  mean={s.mean():.3f}  n_out_of_range={out_of_range}")

if "sentiment_score" in df_val.columns:
    s = df_val["sentiment_score"].dropna()
    out = ((s < -1.5) | (s > 1.5)).sum()
    status = OK if out == 0 else WARN
    print(f"\n  {status} sentiment_score  min={s.min():.3f}  max={s.max():.3f}  mean={s.mean():.3f}  n_out_of_range={out}")

if "timestamp" in df_val.columns:
    ts = df_val["timestamp"]
    print(f"\n  {OK} timestamp  dtype={ts.dtype}  min={ts.min()}  max={ts.max()}")
    if ts.isna().any():
        print(f"  {FAIL} timestamp has {ts.isna().sum()} NaN values — temporal split will fail")

print(f"\n  --- id column types ---")
print(f"  {OK} user_id dtype: {df_val['user_id'].dtype}")
print(f"  {OK} item_id dtype: {df_val['item_id'].dtype}")

counts = df_val.groupby("user_id").size()
few = (counts < 3).sum()
if few > 0:
    print(f"\n  {WARN} {few} users have < 3 interactions — they will be dropped in the split")
else:
    print(f"\n  {OK} All users have >= 3 interactions")

item_cov = df_val.groupby("item_id")[NLI_COLS[0]].apply(lambda x: x.notna().any()) if NLI_COLS and NLI_COLS[0] in df_val.columns else None
if item_cov is not None:
    pct = item_cov.mean() * 100
    status = OK if pct > 70 else WARN
    print(f"  {status} Items with >= 1 valid NLI score for '{NLI_COLS[0]}': {pct:.1f}%")

In [4]:
# load data & temporal split 

candidates = list(Path("/kaggle/input").rglob(DATA_FILE))
DATA_PATH = candidates[0]

df = df_val.copy()
print(f"Rows: {len(df):,} | Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")

# mappings
user2idx = {uid: i for i, uid in enumerate(df["user_id"].unique())}
item2idx = {iid: i for i, iid in enumerate(df["item_id"].unique())}
idx2item = {v: k for k, v in item2idx.items()}
n_users  = len(user2idx)
n_items  = len(item2idx)

df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)

# temporal leave-one-out split
df = df.sort_values(["user_id", "timestamp"]).copy()
df["_rank"] = df.groupby("user_id")["timestamp"].rank(method="first", ascending=False)
test_df  = df[df["_rank"] == 1].copy()
val_df   = df[df["_rank"] == 2].copy()
train_df = df[df["_rank"] > 2].copy()
df = df.drop(columns=["_rank"])

val_gt  = val_df.groupby("user_id")["item_id"].apply(set).to_dict()
test_gt = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
seen_items_val  = train_df.groupby("user_id")["item_idx"].apply(set).to_dict()
seen_items_test = pd.concat([train_df, val_df]).groupby("user_id")["item_idx"].apply(set).to_dict()
all_item_idxs   = list(range(n_items))

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Users: {n_users:,} | Items: {n_items:,}")

Rows: 28,733 | Users: 1,746 | Items: 685
Train: 25,241 | Val: 1,746 | Test: 1,746
Users: 1,746 | Items: 685


In [5]:
def build_feature_tensor(item_df, cols):
    values = item_df[cols].to_numpy(dtype=np.float32)
    mask = ~np.isnan(values)
    for j in range(values.shape[1]):
        m = mask[:, j]
        if m.sum() > 0:
            mu = values[m, j].mean()
            sd = values[m, j].std() + 1e-8
            values[m, j] = (values[m, j] - mu) / sd
        values[~m, j] = 0.0
    feat_t = torch.tensor(values, dtype=torch.float32, device=DEVICE)
    mask_t = torch.tensor(mask,   dtype=torch.bool,    device=DEVICE)
    return feat_t, mask_t

# NLI item-level features
item_nli_df = (
    train_df.sort_values(["item_idx", "timestamp"])
            .groupby("item_idx")[NLI_COLS]
            .first()
            .reindex(range(n_items))
)
item_feat_raw, item_mask_raw = build_feature_tensor(item_nli_df, NLI_COLS)
print("NLI feature tensor :", item_feat_raw.shape)

for col in NLI_COLS:
    cov = item_nli_df[col].notna().mean() * 100
    print(f"  coverage {col}: {cov:.1f}%")

# sentiment target (for MTL auxiliary head)
item_sent_series = train_df.groupby("item_idx")["sentiment_score"].mean().reindex(range(n_items))
sent_vals = item_sent_series.to_numpy(dtype=np.float32)
sent_mask = ~np.isnan(sent_vals)
if sent_mask.sum() > 0:
    mu = sent_vals[sent_mask].mean()
    sd = sent_vals[sent_mask].std() + 1e-8
    sent_vals[sent_mask] = (sent_vals[sent_mask] - mu) / sd
sent_vals[~sent_mask] = 0.0
item_sent_target = torch.tensor(sent_vals, dtype=torch.float32, device=DEVICE)
item_sent_mask   = torch.tensor(sent_mask, dtype=torch.bool,    device=DEVICE)
print("Sentiment target    :", item_sent_target.shape)

# item-level sentiment for Sentiment-CF (z-scored scalar)
item_sent_cf = train_df.groupby("item_idx")["sentiment_score"].mean()
item_sent_cf_arr = np.zeros(n_items, dtype=np.float32)
for idx, val in item_sent_cf.items():
    item_sent_cf_arr[idx] = float(val)
mu  = item_sent_cf_arr.mean()
sd  = item_sent_cf_arr.std() + 1e-8
item_sent_cf_arr = (item_sent_cf_arr - mu) / sd
item_sentiment_tensor = torch.tensor(item_sent_cf_arr, dtype=torch.float32).to(DEVICE)
print("Sentiment tensor    :", item_sentiment_tensor.shape)

NLI feature tensor : torch.Size([685, 3])
  coverage nli_raw_gameplay: 98.4%
  coverage nli_raw_story: 99.4%
  coverage nli_raw_graphics: 87.0%
Sentiment target    : torch.Size([685])
Sentiment tensor    : torch.Size([685])


In [6]:
# SBERT item embeddings
from sentence_transformers import SentenceTransformer

sbert_cache = OUT_DIR / f"{DOMAIN}_item_sbert.npy"

if sbert_cache.exists():
    print("Loading cached SBERT embeddings...")
    item_sbert = np.load(sbert_cache)
else:
    print("Computing SBERT embeddings (mean of last 5 train reviews per item)...")
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))
    item_reviews = (
        train_df.sort_values(["item_idx", "timestamp"], ascending=[True, False])
                .groupby("item_idx")["text_combined"]
                .apply(lambda texts: [t for t in texts.fillna("").astype(str).head(5) if t.strip()])
                .to_dict()
    )
    texts_to_encode, item_ptrs = [], []
    for iidx in range(n_items):
        reviews = item_reviews.get(iidx, [])
        s = len(texts_to_encode)
        texts_to_encode.extend(reviews)
        item_ptrs.append((s, len(texts_to_encode)))
    print(f"  Total texts to encode: {len(texts_to_encode):,}")
    all_embs = sbert_model.encode(
        texts_to_encode, batch_size=32, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=False
    ).astype(np.float32) if texts_to_encode else np.zeros((0, 384), dtype=np.float32)
    item_sbert = np.zeros((n_items, 384), dtype=np.float32)
    for iidx, (s, e) in enumerate(item_ptrs):
        if e > s:
            item_sbert[iidx] = all_embs[s:e].mean(axis=0)
    norms = np.linalg.norm(item_sbert, axis=1, keepdims=True) + 1e-8
    item_sbert /= norms
    np.save(sbert_cache, item_sbert)
    del sbert_model; torch.cuda.empty_cache()
    print(f"  Saved to {sbert_cache}")

item_sbert_tensor = torch.tensor(item_sbert, dtype=torch.float32).to(DEVICE)
print(f"SBERT tensor: {item_sbert_tensor.shape}")

Computing SBERT embeddings (mean of last 5 train reviews per item)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Total texts to encode: 3,421


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

  Saved to /kaggle/working/games_item_sbert.npy
SBERT tensor: torch.Size([685, 384])


In [7]:
# metrics

def ndcg_at_k(rec, rel, k):
    if not rel: return 0.0
    dcg  = sum(1/np.log2(i+2) for i, it in enumerate(rec[:k]) if it in rel)
    idcg = sum(1/np.log2(i+2) for i in range(min(len(rel), k)))
    return dcg/idcg if idcg > 0 else 0.0

def recall_at_k(rec, rel, k):
    if not rel: return 0.0
    return len(set(rec[:k]) & rel) / len(rel)

def precision_at_k(rec, rel, k):
    return len(set(rec[:k]) & rel) / k

def hr_at_k(rec, rel, k):
    return 1.0 if set(rec[:k]) & rel else 0.0

def evaluate(recs, gt, k=10):
    ndcgs, recalls, precisions, hrs = [], [], [], []
    for uid, rec_list in recs.items():
        gt_set = gt.get(uid, set())
        if not gt_set: continue
        ndcgs.append(ndcg_at_k(rec_list, gt_set, k))
        recalls.append(recall_at_k(rec_list, gt_set, k))
        precisions.append(precision_at_k(rec_list, gt_set, k))
        hrs.append(hr_at_k(rec_list, gt_set, k))
    return {f"NDCG@{k}": float(np.mean(ndcgs)),
            f"Recall@{k}": float(np.mean(recalls)),
            f"Precision@{k}": float(np.mean(precisions)),
            f"HR@{k}": float(np.mean(hrs)),
            "n_users": len(ndcgs)}

# BPR dataset
class BPRDataset(TorchDataset):
    def __init__(self, df, n_items):
        df = df[(df["user_idx"] >= 0) & (df["item_idx"] >= 0)].copy()
        self.users = df["user_idx"].values
        self.pos   = df["item_idx"].values
        self.n_items = n_items
        self.user_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()
    def __len__(self): return len(self.users)
    def __getitem__(self, idx):
        u, p = self.users[idx], self.pos[idx]
        seen = self.user_items.get(u, set())
        n = random.randint(0, self.n_items-1)
        while n in seen: n = random.randint(0, self.n_items-1)
        return (torch.tensor(u, dtype=torch.long),
                torch.tensor(p, dtype=torch.long),
                torch.tensor(n, dtype=torch.long))

print("Metrics and dataset defined.")

Metrics and dataset defined.


In [ ]:
# model definitions

# NCF base
class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, mlp_layers, dropout):
        super().__init__()
        self.gmf_u = nn.Embedding(n_users, emb_dim)
        self.gmf_i = nn.Embedding(n_items, emb_dim)
        self.mlp_u = nn.Embedding(n_users, emb_dim)
        self.mlp_i = nn.Embedding(n_items, emb_dim)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_layers:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(emb_dim + mlp_layers[-1], 1)
        for emb in [self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, users, items):
        gmf = self.gmf_u(users) * self.gmf_i(items)
        mlp = self.mlp(torch.cat([self.mlp_u(users), self.mlp_i(items)], dim=-1))
        return self.out(torch.cat([gmf, mlp], dim=-1)).squeeze(-1)

# CF+SBERT
class CFSbert(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, mlp_layers, dropout, sbert_dim):
        super().__init__()
        self.gmf_u = nn.Embedding(n_users, emb_dim)
        self.gmf_i = nn.Embedding(n_items, emb_dim)
        self.mlp_u = nn.Embedding(n_users, emb_dim)
        self.mlp_i = nn.Embedding(n_items, emb_dim)
        self.Wsem  = nn.Linear(sbert_dim, emb_dim, bias=False)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_layers:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(emb_dim + mlp_layers[-1], 1)
        for emb in [self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, users, items, sbert_tensor):
        gmf  = self.gmf_u(users) * self.gmf_i(items)
        qi   = self.mlp_i(items) + self.Wsem(sbert_tensor[items])
        mlp  = self.mlp(torch.cat([self.mlp_u(users), qi], dim=-1))
        return self.out(torch.cat([gmf, mlp], dim=-1)).squeeze(-1)

# Sentiment-CF
class SentimentCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, mlp_layers, dropout):
        super().__init__()
        self.gmf_u = nn.Embedding(n_users, emb_dim)
        self.gmf_i = nn.Embedding(n_items, emb_dim)
        self.mlp_u = nn.Embedding(n_users, emb_dim)
        self.mlp_i = nn.Embedding(n_items, emb_dim)
        layers, in_dim = [], emb_dim * 2 + 1
        for out_dim in mlp_layers:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(emb_dim + mlp_layers[-1], 1)
        for emb in [self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, users, items, sent_tensor):
        gmf  = self.gmf_u(users) * self.gmf_i(items)
        sent = sent_tensor[items].unsqueeze(-1)
        mlp  = self.mlp(torch.cat([self.mlp_u(users), self.mlp_i(items), sent], dim=-1))
        return self.out(torch.cat([gmf, mlp], dim=-1)).squeeze(-1)

# LightGCN
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim, n_layers):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward_full(self, norm_adj):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)
        all_emb = torch.stack(embs).mean(0)
        return all_emb[:self.n_users], all_emb[self.n_users:]

def build_norm_adj(train_df, n_users, n_items):
    users = train_df["user_idx"].values
    items = train_df["item_idx"].values + n_users
    rows  = np.concatenate([users, items])
    cols  = np.concatenate([items, users])
    vals  = np.ones(len(rows), dtype=np.float32)
    n     = n_users + n_items
    adj   = sp.coo_matrix((vals, (rows, cols)), shape=(n, n))
    deg   = np.array(adj.sum(1)).flatten()
    d_inv = np.where(deg > 0, deg**-0.5, 0.0)
    D     = sp.diags(d_inv)
    norm  = (D @ adj @ D).tocoo()
    idx   = torch.tensor(np.stack([norm.row, norm.col]), dtype=torch.long)
    v     = torch.tensor(norm.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(idx, v, (n, n)).coalesce()

# Aspect-CF
class AspectCF(nn.Module):
    def __init__(self, n_users, n_items, n_aspects, emb_dim, asp_dim, mlp_layers, dropout):
        super().__init__()
        self.gmf_u  = nn.Embedding(n_users, emb_dim)
        self.gmf_i  = nn.Embedding(n_items, emb_dim)
        self.mlp_u  = nn.Embedding(n_users, emb_dim)
        self.mlp_i  = nn.Embedding(n_items, emb_dim)
        self.h_asp  = nn.Embedding(n_users, asp_dim)  # user query
        self.asp_emb = nn.Embedding(n_aspects, asp_dim)
        self.W_asp  = nn.Linear(asp_dim, emb_dim, bias=False)
        self.norm   = nn.LayerNorm(emb_dim)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_layers:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(emb_dim + mlp_layers[-1], 1)
        for emb in [self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(emb.weight, std=0.01)
        nn.init.normal_(self.h_asp.weight, std=0.01)
        nn.init.normal_(self.asp_emb.weight, std=0.01)

    def _aspect_repr(self, users, items, feat, mask):
        B = users.shape[0]
        n_asp = feat.shape[1]
        scores = feat[items]
        m      = mask[items]
        asp_id = torch.arange(n_asp, device=DEVICE)
        t      = self.asp_emb(asp_id).unsqueeze(0) * scores.unsqueeze(-1)
        q      = self.h_asp(users).unsqueeze(1)
        logits = (q * t).sum(-1) / (t.shape[-1]**0.5)
        logits = logits.masked_fill(~m, -1e9)
        alpha  = torch.softmax(logits, dim=-1) 
        alpha  = alpha * m.float()
        z      = (alpha.unsqueeze(-1) * t).sum(1)
        return self.norm(self.W_asp(z))

    def forward_rank(self, users, items, feat, mask):
        gmf  = self.gmf_u(users) * self.gmf_i(items)
        z    = self._aspect_repr(users, items, feat, mask)
        qi   = self.mlp_i(items) + z
        mlp  = self.mlp(torch.cat([self.mlp_u(users), qi], dim=-1))
        return self.out(torch.cat([gmf, mlp], dim=-1)).squeeze(-1)

# Aspect-MTL
class AspectMTL(nn.Module):
    def __init__(self, base: AspectCF):
        super().__init__()
        self.base    = base
        emb_dim      = base.mlp_i.embedding_dim
        mlp_out_dim  = base.out.in_features - emb_dim
        self.sent_head = nn.Sequential(
            nn.Linear(emb_dim + mlp_out_dim, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward_rank(self, users, items, feat, mask):
        return self.base.forward_rank(users, items, feat, mask)

    def forward_aux(self, items, feat, mask):
        dummy_u = torch.zeros_like(items)
        gmf   = self.base.gmf_u(dummy_u) * self.base.gmf_i(items)
        z     = self.base._aspect_repr(dummy_u, items, feat, mask)
        qi    = self.base.mlp_i(items) + z
        mlp   = self.base.mlp(torch.cat([self.base.mlp_u(dummy_u), qi], dim=-1))
        h     = torch.cat([gmf, mlp], dim=-1)
        return self.sent_head(h).squeeze(-1)

In [ ]:
# shared training loop helpers

def bpr_loss_fn(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

@torch.no_grad()
def gen_recs_ncf_family(model, forward_fn, users_list, seen_dict):
    model.eval()
    recs = {}
    all_i = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
    for uid in tqdm(users_list, desc="Recs", leave=False):
        uidx = user2idx.get(uid, -1)
        if uidx < 0: continue
        u_t  = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
        sc   = forward_fn(u_t, all_i).cpu().numpy()
        for sidx in seen_dict.get(uid, set()): sc[sidx] = -1e9
        top  = np.argsort(sc)[::-1][:K]
        recs[uid] = [idx2item[i] for i in top]
    return recs

def train_ncf_family(model, forward_fn, name, lr=LR_SCRATCH):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    dataset   = BPRDataset(train_df, n_items)
    loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_users = val_df["user_id"].unique().tolist()

    best_ndcg, no_improve, best_state = 0.0, 0, None
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for users_b, pos_b, neg_b in tqdm(loader, desc=f"Ep {epoch}", leave=False):
            users_b = users_b.to(DEVICE)
            pos_b   = pos_b.to(DEVICE)
            neg_b   = neg_b.to(DEVICE)
            loss = bpr_loss_fn(forward_fn(users_b, pos_b), forward_fn(users_b, neg_b))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(users_b)

        if epoch % 2 == 0 or epoch == 1:
            model.eval()
            sample_users = random.sample(val_users, min(2000, len(val_users)))
            val_recs_q   = gen_recs_ncf_family(model, forward_fn, sample_users, seen_items_val)
            val_ndcg     = np.mean([ndcg_at_k(v, val_gt[u], K) for u, v in val_recs_q.items() if u in val_gt])
            history.append({"epoch": epoch, "val_ndcg": float(val_ndcg)})
            print(f"  Epoch {epoch:>3}/{MAX_EPOCHS}  BPR: {total_loss/len(dataset):.4f}  val_NDCG@10: {val_ndcg:.4f}")

            if val_ndcg > best_ndcg + 1e-5:
                best_ndcg  = val_ndcg
                no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"  Early stopping (best: {best_ndcg:.4f})")
                    break

    if best_state:
        model.load_state_dict(best_state)
        model.to(DEVICE)

    test_users = test_df["user_id"].unique().tolist()
    val_recs   = gen_recs_ncf_family(model, forward_fn, val_users,  seen_items_val)
    test_recs  = gen_recs_ncf_family(model, forward_fn, test_users, seen_items_test)
    val_m  = evaluate(val_recs,  val_gt,  K)
    test_m = evaluate(test_recs, test_gt, K)
    print(f"\n  {name} — VAL  NDCG@10={val_m['NDCG@10']:.4f}  HR@10={val_m['HR@10']:.4f}")
    print(f"  {name} — TEST NDCG@10={test_m['NDCG@10']:.4f}  HR@10={test_m['HR@10']:.4f}")
    torch.save(model.state_dict(), OUT_DIR / f"{DOMAIN}_{name.lower().replace('+','').replace(' ','_')}_model.pt")
    return val_m, test_m, history

In [10]:
# NCF
print("="*50)
print("Training NCF...")
print("="*50)

ncf_model = NCF(n_users, n_items, EMB_DIM, MLP_LAYERS, DROPOUT).to(DEVICE)

def ncf_forward(u, i): return ncf_model(u, i)

ncf_val, ncf_test, _ = train_ncf_family(ncf_model, ncf_forward, "NCF")
ncf_results = {"model": "NCF", "val": ncf_val, "test": ncf_test}

Training NCF...


  Epoch   1/50  BPR: 0.6777  val_NDCG@10: 0.0130


  Epoch   2/50  BPR: 0.6248  val_NDCG@10: 0.0109


  Epoch   4/50  BPR: 0.5447  val_NDCG@10: 0.0234


  Epoch   6/50  BPR: 0.4075  val_NDCG@10: 0.0360


  Epoch   8/50  BPR: 0.3097  val_NDCG@10: 0.0334


  Epoch  10/50  BPR: 0.2687  val_NDCG@10: 0.0346


  Epoch  12/50  BPR: 0.2356  val_NDCG@10: 0.0338


  Epoch  14/50  BPR: 0.2201  val_NDCG@10: 0.0356
  Early stopping (best: 0.0360)



  NCF — VAL  NDCG@10=0.0360  HR@10=0.0779
  NCF — TEST NDCG@10=0.0317  HR@10=0.0687


In [11]:
# LightGCN
print("="*50)
print("Training LightGCN...")
print("="*50)

norm_adj = build_norm_adj(train_df, n_users, n_items).to(DEVICE)
lgcn_model = LightGCN(n_users, n_items, EMB_DIM*2, LGCN_LAYERS).to(DEVICE)
optimizer_lgcn = torch.optim.Adam(lgcn_model.parameters(), lr=LR_SCRATCH, weight_decay=WEIGHT_DECAY)
dataset_lgcn   = BPRDataset(train_df, n_items)
loader_lgcn    = DataLoader(dataset_lgcn, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_users_lgcn = val_df["user_id"].unique().tolist()
test_users_lgcn = test_df["user_id"].unique().tolist()

best_ndcg_l, no_improve_l, best_state_l = 0.0, 0, None

for epoch in range(1, MAX_EPOCHS+1):
    lgcn_model.train()
    total_l = 0.0
    for u_b, p_b, n_b in tqdm(loader_lgcn, desc=f"Ep {epoch}", leave=False):
        u_b, p_b, n_b = u_b.to(DEVICE), p_b.to(DEVICE), n_b.to(DEVICE)
        u_emb, i_emb = lgcn_model.forward_full(norm_adj)
        pos_s = (u_emb[u_b] * i_emb[p_b]).sum(-1)
        neg_s = (u_emb[u_b] * i_emb[n_b]).sum(-1)
        loss  = bpr_loss_fn(pos_s, neg_s)
        optimizer_lgcn.zero_grad(); loss.backward(); optimizer_lgcn.step()
        total_l += loss.item() * len(u_b)

    if epoch % 2 == 0 or epoch == 1:
        lgcn_model.eval()
        with torch.no_grad():
            u_emb, i_emb = lgcn_model.forward_full(norm_adj)
        sample_u = random.sample(val_users_lgcn, min(2000, len(val_users_lgcn)))
        ndcg_vals = []
        for uid in sample_u:
            uidx = user2idx.get(uid, -1)
            if uidx < 0: continue
            sc = (u_emb[uidx] @ i_emb.T).cpu().numpy()
            for sidx in seen_items_val.get(uid, set()): sc[sidx] = -1e9
            top = [idx2item[i] for i in np.argsort(sc)[::-1][:K]]
            ndcg_vals.append(ndcg_at_k(top, val_gt[uid], K))
        val_ndcg_l = float(np.mean(ndcg_vals)) if ndcg_vals else 0.0
        print(f"  Epoch {epoch:>3}  BPR: {total_l/len(dataset_lgcn):.4f}  val_NDCG@10: {val_ndcg_l:.4f}")
        if val_ndcg_l > best_ndcg_l + 1e-5:
            best_ndcg_l = val_ndcg_l; no_improve_l = 0
            best_state_l = {k: v.detach().cpu().clone() for k, v in lgcn_model.state_dict().items()}
        else:
            no_improve_l += 1
            if no_improve_l >= PATIENCE:
                print(f"  Early stopping (best: {best_ndcg_l:.4f})"); break

if best_state_l:
    lgcn_model.load_state_dict(best_state_l); lgcn_model.to(DEVICE)

@torch.no_grad()
def lgcn_recs(users_list, seen_dict):
    lgcn_model.eval()
    u_emb, i_emb = lgcn_model.forward_full(norm_adj)
    recs = {}
    for uid in tqdm(users_list, desc="Recs", leave=False):
        uidx = user2idx.get(uid, -1)
        if uidx < 0: continue
        sc = (u_emb[uidx] @ i_emb.T).cpu().numpy()
        for sidx in seen_dict.get(uid, set()): sc[sidx] = -1e9
        top = [idx2item[i] for i in np.argsort(sc)[::-1][:K]]
        recs[uid] = top
    return recs

lgcn_val_m  = evaluate(lgcn_recs(val_users_lgcn, seen_items_val),   val_gt,  K)
lgcn_test_m = evaluate(lgcn_recs(test_users_lgcn, seen_items_test), test_gt, K)
print(f"\n  LightGCN — VAL  NDCG@10={lgcn_val_m['NDCG@10']:.4f}  HR@10={lgcn_val_m['HR@10']:.4f}")
print(f"  LightGCN — TEST NDCG@10={lgcn_test_m['NDCG@10']:.4f}  HR@10={lgcn_test_m['HR@10']:.4f}")
torch.save(lgcn_model.state_dict(), OUT_DIR / f"{DOMAIN}_lightgcn_model.pt")
lgcn_results = {"model": "LightGCN", "val": lgcn_val_m, "test": lgcn_test_m}

Training LightGCN...


  Epoch   1  BPR: 0.6869  val_NDCG@10: 0.0349


  Epoch   2  BPR: 0.6013  val_NDCG@10: 0.0327


  Epoch   4  BPR: 0.3912  val_NDCG@10: 0.0326


  Epoch   6  BPR: 0.3261  val_NDCG@10: 0.0318


  Epoch   8  BPR: 0.2997  val_NDCG@10: 0.0315
  Early stopping (best: 0.0349)



  LightGCN — VAL  NDCG@10=0.0349  HR@10=0.0756
  LightGCN — TEST NDCG@10=0.0269  HR@10=0.0641


In [12]:
# CF+SBERT
print("="*50)
print("Training CF+SBERT...")
print("="*50)

sbert_dim  = item_sbert_tensor.shape[1]
sbert_model = CFSbert(n_users, n_items, EMB_DIM, MLP_LAYERS, DROPOUT, sbert_dim).to(DEVICE)

def sbert_forward(u, i): return sbert_model(u, i, item_sbert_tensor)

sbert_val, sbert_test, _ = train_ncf_family(sbert_model, sbert_forward, "CF+SBERT")
sbert_results = {"model": "CF+SBERT", "val": sbert_val, "test": sbert_test}

Training CF+SBERT...


  Epoch   1/50  BPR: 0.6732  val_NDCG@10: 0.0096


  Epoch   2/50  BPR: 0.6263  val_NDCG@10: 0.0107


  Epoch   4/50  BPR: 0.5492  val_NDCG@10: 0.0197


  Epoch   6/50  BPR: 0.3946  val_NDCG@10: 0.0286


...

  Epoch  26/50  BPR: 0.1564  val_NDCG@10: 0.0372


  Epoch  28/50  BPR: 0.1469  val_NDCG@10: 0.0381


  Epoch  30/50  BPR: 0.1412  val_NDCG@10: 0.0385


  Epoch  32/50  BPR: 0.1315  val_NDCG@10: 0.0402
  Early stopping (best: 0.0404)



  CF+SBERT — VAL  NDCG@10=0.0404  HR@10=0.0842
  CF+SBERT — TEST NDCG@10=0.0365  HR@10=0.0716


In [13]:
# Sentiment-CF
print("="*50)
print("Training Sentiment-CF...")
print("="*50)

sentcf_model = SentimentCF(n_users, n_items, EMB_DIM, MLP_LAYERS, DROPOUT).to(DEVICE)

def sentcf_forward(u, i): return sentcf_model(u, i, item_sentiment_tensor)

sentcf_val, sentcf_test, _ = train_ncf_family(sentcf_model, sentcf_forward, "Sentiment-CF")
sentcf_results = {"model": "Sentiment-CF", "val": sentcf_val, "test": sentcf_test}

Training Sentiment-CF...


  Epoch   1/50  BPR: 0.6778  val_NDCG@10: 0.0120


  Epoch   2/50  BPR: 0.6224  val_NDCG@10: 0.0104


  Epoch   4/50  BPR: 0.5301  val_NDCG@10: 0.0205


  Epoch   6/50  BPR: 0.3786  val_NDCG@10: 0.0336


...

  Epoch  24/50  BPR: 0.1493  val_NDCG@10: 0.0401


  Epoch  26/50  BPR: 0.1404  val_NDCG@10: 0.0374


  Epoch  28/50  BPR: 0.1325  val_NDCG@10: 0.0405


  Epoch  30/50  BPR: 0.1279  val_NDCG@10: 0.0376
  Early stopping (best: 0.0409)



  Sentiment-CF — VAL  NDCG@10=0.0409  HR@10=0.0819
  Sentiment-CF — TEST NDCG@10=0.0333  HR@10=0.0664


In [14]:
# ── Aspect-CF
print("="*50)
print("Training Aspect-CF...")
print("="*50)

n_aspects   = len(NLI_COLS)
asp_model   = AspectCF(n_users, n_items, n_aspects, EMB_DIM, ASP_DIM, MLP_LAYERS, DROPOUT).to(DEVICE)

def asp_forward(u, i): return asp_model.forward_rank(u, i, item_feat_raw, item_mask_raw)

asp_val, asp_test, _ = train_ncf_family(asp_model, asp_forward, "Aspect-CF")
asp_results = {"model": "Aspect-CF", "val": asp_val, "test": asp_test}

Training Aspect-CF...


  Epoch   1/50  BPR: 0.6884  val_NDCG@10: 0.0088


  Epoch   2/50  BPR: 0.6678  val_NDCG@10: 0.0121


  Epoch   4/50  BPR: 0.5394  val_NDCG@10: 0.0229


  Epoch   6/50  BPR: 0.4001  val_NDCG@10: 0.0292


...

  Epoch  38/50  BPR: 0.1107  val_NDCG@10: 0.0412


  Epoch  40/50  BPR: 0.1070  val_NDCG@10: 0.0397


  Epoch  42/50  BPR: 0.1022  val_NDCG@10: 0.0394


  Epoch  44/50  BPR: 0.0928  val_NDCG@10: 0.0403
  Early stopping (best: 0.0426)



  Aspect-CF — VAL  NDCG@10=0.0426  HR@10=0.0876
  Aspect-CF — TEST NDCG@10=0.0341  HR@10=0.0676


In [15]:
# Aspect-MTL (warm-start from Aspect-CF)
print("="*50)
print("Training Aspect-MTL (warm-start from Aspect-CF)...")
print("="*50)

import copy
asp_backbone = copy.deepcopy(asp_model)   # warm-start from best Aspect-CF weights
mtl_model    = AspectMTL(asp_backbone).to(DEVICE)

optimizer_mtl = torch.optim.Adam(mtl_model.parameters(), lr=LR_WARMSTART, weight_decay=WEIGHT_DECAY)
dataset_mtl   = BPRDataset(train_df, n_items)
loader_mtl    = DataLoader(dataset_mtl, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_users_mtl  = val_df["user_id"].unique().tolist()
test_users_mtl = test_df["user_id"].unique().tolist()

best_ndcg_m, no_improve_m, best_state_m = 0.0, 0, None

for epoch in range(1, MAX_EPOCHS+1):
    mtl_model.train()
    total_bpr = total_sent = 0.0
    for u_b, p_b, n_b in tqdm(loader_mtl, desc=f"Ep {epoch}", leave=False):
        u_b, p_b, n_b = u_b.to(DEVICE), p_b.to(DEVICE), n_b.to(DEVICE)
        pos_s = mtl_model.forward_rank(u_b, p_b, item_feat_raw, item_mask_raw)
        neg_s = mtl_model.forward_rank(u_b, n_b, item_feat_raw, item_mask_raw)
        bpr   = bpr_loss_fn(pos_s, neg_s)
        # auxiliary sentiment loss on positive items
        pos_u = p_b.unique()
        sp    = mtl_model.forward_aux(pos_u, item_feat_raw, item_mask_raw)
        m     = item_sent_mask[pos_u]
        sent_loss = ((sp[m] - item_sent_target[pos_u][m])**2).mean() if m.any() else torch.tensor(0.0, device=DEVICE)
        loss  = bpr + LAMBDA_SENT * sent_loss
        optimizer_mtl.zero_grad(); loss.backward(); optimizer_mtl.step()
        total_bpr  += bpr.item()       * len(u_b)
        total_sent += sent_loss.item() * len(u_b)

    if epoch % 2 == 0 or epoch == 1:
        mtl_model.eval()
        sample_u = random.sample(val_users_mtl, min(2000, len(val_users_mtl)))
        ndcg_vals = []
        with torch.no_grad():
            for uid in sample_u:
                uidx = user2idx.get(uid, -1)
                if uidx < 0: continue
                all_i = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
                u_t   = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
                sc    = mtl_model.forward_rank(u_t, all_i, item_feat_raw, item_mask_raw).cpu().numpy()
                for sidx in seen_items_val.get(uid, set()): sc[sidx] = -1e9
                top = [idx2item[i] for i in np.argsort(sc)[::-1][:K]]
                ndcg_vals.append(ndcg_at_k(top, val_gt[uid], K))
        val_ndcg_m = float(np.mean(ndcg_vals)) if ndcg_vals else 0.0
        n = len(dataset_mtl)
        print(f"  Epoch {epoch:>3}  BPR: {total_bpr/n:.4f}  Sent: {total_sent/n:.4f}  val_NDCG@10: {val_ndcg_m:.4f}")
        if val_ndcg_m > best_ndcg_m + 1e-5:
            best_ndcg_m = val_ndcg_m; no_improve_m = 0
            best_state_m = {k: v.detach().cpu().clone() for k, v in mtl_model.state_dict().items()}
        else:
            no_improve_m += 1
            if no_improve_m >= PATIENCE:
                print(f"  Early stopping (best: {best_ndcg_m:.4f})"); break

if best_state_m:
    mtl_model.load_state_dict(best_state_m); mtl_model.to(DEVICE)

@torch.no_grad()
def mtl_recs(users_list, seen_dict):
    mtl_model.eval()
    all_i = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
    recs = {}
    for uid in tqdm(users_list, desc="Recs", leave=False):
        uidx = user2idx.get(uid, -1)
        if uidx < 0: continue
        u_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
        sc  = mtl_model.forward_rank(u_t, all_i, item_feat_raw, item_mask_raw).cpu().numpy()
        for sidx in seen_dict.get(uid, set()): sc[sidx] = -1e9
        top = [idx2item[i] for i in np.argsort(sc)[::-1][:K]]
        recs[uid] = top
    return recs

mtl_val_m  = evaluate(mtl_recs(val_users_mtl, seen_items_val),   val_gt,  K)
mtl_test_m = evaluate(mtl_recs(test_users_mtl, seen_items_test), test_gt, K)
print(f"\n  Aspect-MTL — VAL  NDCG@10={mtl_val_m['NDCG@10']:.4f}  HR@10={mtl_val_m['HR@10']:.4f}")
print(f"  Aspect-MTL — TEST NDCG@10={mtl_test_m['NDCG@10']:.4f}  HR@10={mtl_test_m['HR@10']:.4f}")
torch.save(mtl_model.state_dict(), OUT_DIR / f"{DOMAIN}_aspect_mtl_model.pt")
mtl_results = {"model": "Aspect-MTL", "val": mtl_val_m, "test": mtl_test_m}

Training Aspect-MTL (warm-start from Aspect-CF)...


  Epoch   1  BPR: 0.1142  Sent: 0.7941  val_NDCG@10: 0.0428


  Epoch   2  BPR: 0.1140  Sent: 0.7108  val_NDCG@10: 0.0426


  Epoch   4  BPR: 0.1087  Sent: 0.6084  val_NDCG@10: 0.0421


  Epoch   6  BPR: 0.1064  Sent: 0.5490  val_NDCG@10: 0.0399


  Epoch   8  BPR: 0.1013  Sent: 0.5412  val_NDCG@10: 0.0401
  Early stopping (best: 0.0428)



  Aspect-MTL — VAL  NDCG@10=0.0428  HR@10=0.0888
  Aspect-MTL — TEST NDCG@10=0.0355  HR@10=0.0699


In [16]:
# Final Results Table

all_res = [
    ncf_results,
    lgcn_results,
    sbert_results,
    sentcf_results,
    asp_results,
    mtl_results,
]

rows = []
for r in all_res:
    rows.append({
        "Model":          r["model"],
        "Val NDCG@10":    r["val"]["NDCG@10"],
        "Test NDCG@10":   r["test"]["NDCG@10"],
        "Test HR@10":     r["test"]["HR@10"],
        "Test Recall@10": r["test"]["Recall@10"],
    })

results_df = pd.DataFrame(rows).sort_values("Val NDCG@10", ascending=False).reset_index(drop=True)

csv_path = OUT_DIR / f"{DOMAIN}_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

json_path = OUT_DIR / f"{DOMAIN}_results.json"
with open(json_path, "w") as f:
    json.dump([r for r in all_res], f, indent=2)
print(f"Saved: {json_path}")

ncf_test = results_df.loc[results_df["Model"]=="NCF", "Test NDCG@10"].values[0]
best_val  = results_df["Val NDCG@10"].max()
best_test = results_df["Test NDCG@10"].max()
best_hr   = results_df["Test HR@10"].max()

html = f"<h3>{DOMAIN.upper()} — Recommendation Results</h3>"
html += "<table style='border-collapse:collapse; font-size:14px; font-family:monospace;'>"
html += "<tr style='border-bottom:2px solid black;'>"
for col in ["Model", "Val NDCG@10", "Test NDCG@10", "Test HR@10", "Test Recall@10", "Δ vs NCF"]:
    html += f"<th style='padding:8px 14px; text-align:center;'>{col}</th>"
html += "</tr>"

for _, row in results_df.iterrows():
    html += "<tr>"
    html += f"<td style='padding:7px 14px; text-align:left;'>{row['Model']}</td>"

    for col, best in [("Val NDCG@10", best_val), ("Test NDCG@10", best_test), ("Test HR@10", best_hr), ("Test Recall@10", None)]:
        v   = row[col]
        fmt = f"<b>{v:.4f}</b>" if best and abs(v - best) < 1e-9 else f"{v:.4f}"
        html += f"<td style='padding:7px 14px; text-align:center; border-bottom:1px solid #ddd;'>{fmt}</td>"

    delta = (row["Test NDCG@10"] / ncf_test - 1) * 100 if ncf_test > 0 else 0
    sign  = "+" if delta > 0 else ""
    color = "#2a9d2a" if delta > 0 else ("#c0392b" if delta < -0.5 else "#555")
    html += f"<td style='padding:7px 14px; text-align:center; border-bottom:1px solid #ddd; color:{color};'>{sign}{delta:.1f}%</td>"
    html += "</tr>"

html += "</table>"
display(HTML(html))

print("\nRaw numbers:")
print(results_df.to_string(index=False))

Saved: /kaggle/working/games_results.csv
Saved: /kaggle/working/games_results.json


Model,Val NDCG@10,Test NDCG@10,Test HR@10,Test Recall@10,Δ vs NCF
Aspect-MTL,0.0428,0.0355,0.0699,0.0699,+12.2%
Aspect-CF,0.0426,0.0341,0.0676,0.0676,+7.8%
Sentiment-CF,0.0409,0.0333,0.0664,0.0664,+5.1%
CF+SBERT,0.0404,0.0365,0.0716,0.0716,+15.4%
NCF,0.0360,0.0317,0.0687,0.0687,0.0%
LightGCN,0.0349,0.0269,0.0641,0.0641,-15.0%



Raw numbers:
       Model  Val NDCG@10  Test NDCG@10  Test HR@10  Test Recall@10
  Aspect-MTL     0.042756      0.035511    0.069874        0.069874
   Aspect-CF     0.042613      0.034112    0.067583        0.067583
Sentiment-CF     0.040867      0.033261    0.066438        0.066438
    CF+SBERT     0.040387      0.036540    0.071592        0.071592
         NCF     0.035971      0.031654    0.068729        0.068729
    LightGCN     0.034927      0.026892    0.064147        0.064147


In [17]:
print(item_nli_df.notna().mean())

nli_raw_gameplay    0.983942
nli_raw_story       0.994161
nli_raw_graphics    0.870073
dtype: float64
